In [ ]:
import math
import locale
import numpy as np
import mip
from collections import namedtuple
import glob
import pandas as pd
from IPython.display import display, Markdown

In [ ]:
##### Open the main sheet #####

directory = "examples/Mariners"
globFilename = directory + "/Full Name*.xls*"
excelFiles = glob.glob(globFilename)
if len(excelFiles) < 1 or len(excelFiles) > 1:
    print(f"Can't find unique excel file: {globFilename}")
    exit()
mainSheet = pd.read_excel(excelFiles[0])

In [ ]:
##### Find the constraint row #####
for constraintRow in range(len(mainSheet.Date)):
    if mainSheet.iat[constraintRow,2] == "At least":
        break

In [ ]:
##### Establish an object for each day of the season
 
locale.setlocale(locale.LC_ALL, '')
Game = namedtuple('Game', ('weekday', 'month', 'day', 'gameDay', 'time', 'opponent', 'type', 'price', 'pairs', 'seats'))

##### Read the season schedule and store it as a list of games
 
openingDay = mainSheet.Date[0]
schedule = []
GamesInPlan = 0
PairsInPlan = 0
MaxPairsPerGame = 0
for weekday, date, time, opponent, type, price, pairs, seats in zip(mainSheet.Day, mainSheet.Date, mainSheet.Time, mainSheet.Opponent, mainSheet.Type, mainSheet.Price, mainSheet.GamePairs, mainSheet.Seats):
    if not isinstance(weekday, str) or weekday == "" or pd.isna(date):
        break
    schedule.append(Game(weekday, date.month, date.day, (date - openingDay).days, time.strftime("%I:%M %p"), opponent, type, price, pairs, seats))
    GamesInPlan += 1
    PairsInPlan += pairs
    MaxPairsPerGame = max(MaxPairsPerGame, pairs)


In [ ]:
#########################################################
# Build a dictionary out of a list of games
#
#     buildCode == 0:    Create constraints for pair and quads
#     buildCode == 2:    Create constraint for pairs only
#     buildCode == 4:    Create constraint for quads only

def BuildDict(gameList, buildCode = 0):
    pairList = []
    for game in gameList:
        if buildCode != 4:
            pairList.append((game, 1.0))
        if buildCode != 2:
            pairList.append((game + GamesInPlan, 1.0))
    return dict(pairList)

# Define a class to contain constraints

class Constraint:
    def __init__(self, description, isChecked, comparator, value, gameDictionaries = None):
        self.description = description
        self.isChecked = isChecked
        self.comparator = comparator
        self.value = value
        self.gameDictionaries = list() if gameDictionaries is None else gameDictionaries

# Define a class to contain each person's preferences

class SportsFan:
    def __init__(self, name, pairs, quads, ranking, extra = None):
        self.name = name
        self.pairs = pairs
        self.quads = quads

# Assign weights to games

        if len(ranking) != 0:
            self.ranking = ranking
        else:
            self.ranking = GamesInPlan * [GamesInPlan // 2]

# Adjust weights to favor highly ranked games

        self.useRanking = []
        midpoint = (GamesInPlan - 1) // 2
        for wgt in self.ranking:
            if wgt <= midpoint + 1:
                self.useRanking.append(math.sqrt(wgt - 1.0))
            else:
                self.useRanking.append(2.0 * math.sqrt(midpoint) - math.sqrt(2.0 * midpoint - wgt + 1))
        if len(self.useRanking) == GamesInPlan:
            self.useRanking += [2.0 * cost for cost in self.useRanking]
        else:
            for ix in range(GamesInPlan):
                self.useRanking[GamesInPlan + ix] *= 2

# Each person must attend correct number of games

        pairCon = Constraint(f"{self.name}: Pairs = {self.pairs}", True, '==', self.pairs, [dict([(ix, 1.0) for ix in range(GamesInPlan)])])
        quadCon = Constraint(f"{self.name}: Quads = {self.quads}", True, '==', self.quads, [dict([(ix, 1.0) for ix in range(GamesInPlan, 2 * GamesInPlan)])])

# Save all of the constraints for this person

        self.constraints = [pairCon, quadCon]
        if extra is not None:
            self.constraints += extra

##### This constraint handles the spacing of games

def Spacing(description, checked, comparator, value, pairsOrQuads = 0):
    constraint = Constraint(description, checked, comparator, np.float64(1))
    if not isinstance(value, (int,float,np.floating,np.integer)) or pd.isna(value) or value < 1 or value > GamesInPlan:
        print(f"Bad constraint:{description}")
        return constraint
    daysApart = value + 1
    firstGame = 0
    lastGame = 0
    while True:
        while lastGame < GamesInPlan and schedule[firstGame].gameDay + daysApart > schedule[lastGame].gameDay:
            lastGame += 1
        constraint.gameDictionaries.append(BuildDict(range(firstGame, lastGame), pairsOrQuads))
        if lastGame == GamesInPlan:
            break
        while schedule[firstGame].gameDay + daysApart <= schedule[lastGame].gameDay:
            firstGame += 1
    return constraint

##### Require or forbid games in various months

def Monthly(description, checked, comparator, value, pairsOrQuads = 0):
    constraint = Constraint(description, checked, comparator, value)
    if not isinstance(value, (int,float,np.floating,np.integer)) or pd.isna(value) or value < 1 or value > 30:
        print(f"Bad constraint:{description}")
        return constraint
    firstGame = 0
    lastGame = 0
    while True:
        while schedule[lastGame].month < 5:
            lastGame += 1
        while lastGame < GamesInPlan and schedule[firstGame].month == schedule[lastGame].month:
            lastGame += 1
        constraint.gameDictionaries.append(BuildDict(range(firstGame, lastGame), pairsOrQuads))
        if lastGame == GamesInPlan:
            break
        firstGame = lastGame
    return constraint

##### Require or forbid games in different series

def Series(description, checked, comparator, value, pairsOrQuads = 0):
    constraint = Constraint(description, checked, comparator, value)
    if not isinstance(value, (int,float,np.floating,np.integer)) or pd.isna(value) or value < 1 or value > GamesInPlan:
        print(f"Bad constraint:{description}")
        return constraint
    firstGame = 0
    lastGame = 0
    while True:
        while lastGame < GamesInPlan and schedule[firstGame].opponent == schedule[lastGame].opponent:
            lastGame += 1
        constraint.gameDictionaries.append(BuildDict(range(firstGame, lastGame), pairsOrQuads))
        if lastGame == GamesInPlan:
            break
        firstGame = lastGame
    return constraint

##### Require or forbid games for different opponents

def Opponents(description, checked, comparator, value, pairsOrQuads = 0):
    constraint = Constraint(description, checked, comparator, value)
    if not isinstance(value, (int,float,np.floating,np.integer)) or pd.isna(value) or value < 1 or value > GamesInPlan:
        print(f"Bad constraint:{description}")
        return constraint
    firstGame = 0
    lastGame = 0
    opponentDict = {}
    while True:
        while lastGame < GamesInPlan and schedule[firstGame].opponent == schedule[lastGame].opponent:
            lastGame += 1
        opponentGames = opponentDict.get(schedule[firstGame].opponent, [])
        opponentGames += range(firstGame, lastGame)
        opponentDict[schedule[firstGame].opponent] = opponentGames
        if lastGame == GamesInPlan:
            break
        firstGame = lastGame
    for opponentGames in opponentDict.values():
        constraint.gameDictionaries.append(BuildDict(opponentGames, pairsOrQuads))
    return constraint


In [ ]:
SpreadSheetConstraint = namedtuple('SpreadSheetConstraint', ('function', 'comparator', 'pairsOrQuads'))
constraintsToProcess = [SpreadSheetConstraint(Monthly, '>=', 0),
                 SpreadSheetConstraint(Monthly, '<=', 0),
                 SpreadSheetConstraint(Monthly, '>=', 2),
                 SpreadSheetConstraint(Monthly, '<=', 2),
                 SpreadSheetConstraint(Monthly, '>=', 4),
                 SpreadSheetConstraint(Monthly, '<=', 4),
                 SpreadSheetConstraint(Spacing, '<=', 0),
                 SpreadSheetConstraint(Spacing, '>=', 0),
                 SpreadSheetConstraint(Spacing, '<=', 2),
                 SpreadSheetConstraint(Spacing, '>=', 2),
                 SpreadSheetConstraint(Spacing, '<=', 4),
                 SpreadSheetConstraint(Spacing, '>=', 4),
                 SpreadSheetConstraint(Series, '<=', 0),
                 SpreadSheetConstraint(Opponents, '<=', 0)]

In [ ]:
##### Define the participants here #####

fans = []
totalPairs = 0
for fullName, nPairs, nQuads in zip(mainSheet.FullName, mainSheet.Pairs, mainSheet.Quads):
    if not isinstance(fullName, str) or fullName == "":
        break
    totalPairs += nPairs + 2 * nQuads
    globFilename = directory + "/" + fullName + "*.xls*"
    excelFiles = glob.glob(globFilename)
    if len(excelFiles) < 1 or len(excelFiles) > 1:
        print(f"Can't find unique excel file: {globFilename}")
        continue
    fanSheet = pd.read_excel(excelFiles[0])
    pairsRank = [np.int64(fanSheet.Pick[ix]) for ix in range(GamesInPlan)]
    if not isinstance(fanSheet.QuadPick[0], np.float64) and not math.isnan(fanSheet.QuadPick[0]):
        quadsRank = [np.int64(fanSheet.QuadPick[ix]) for ix in range(GamesInPlan)]
        pairsRank += quadsRank

    # Add fan constraints
    extraConstraints = []
    for ix, sheetConstraint in enumerate(constraintsToProcess):
        row = constraintRow + ix
        if fanSheet.iat[row,1] or not pd.isna(fanSheet.iat[row,3]):
            description = f"{fullName}: {fanSheet.iat[row,2]} {fanSheet.iat[row,3]} {fanSheet.iat[row,4]}"
            extraConstraints.append(sheetConstraint.function(description, fanSheet.iat[row,1], sheetConstraint.comparator, fanSheet.iat[row,3], sheetConstraint.pairsOrQuads))
    fans.append(SportsFan(fullName, nPairs, nQuads, pairsRank, extraConstraints))

leftOver = PairsInPlan - totalPairs
if leftOver < 0:
    print("Too many games requested")
maxSparePairs = max((leftOver * GamesInPlan) // PairsInPlan, 1) # Maximum # of games that could be completely unassigned
while leftOver > 0:
    pairsRank = (GamesInPlan * [np.int64(1)])[:]
    nPairs = min(leftOver, maxSparePairs)
    fans.append(SportsFan(f"Spare Pair", nPairs, 0, pairsRank))
    leftOver -= nPairs


In [ ]:
for gix, game in enumerate(schedule):
    picks = []
    for fan in fans:
        picks.append(int(fan.ranking[gix]))
    print(f"{picks} {game.weekday} {game.month}/{game.day} {game.opponent}{game.time} {game.type} {game.seats}")

In [ ]:
try:
    tixModel = mip.Model()
    tixVars = []
    for fan in fans:
        tixVars += [tixModel.add_var(name = fan.name + f"_pair_game_{ix}", var_type = mip.BINARY) for ix in range(GamesInPlan)]
        tixVars += [tixModel.add_var(name = fan.name + f"_quad_game_{ix}", var_type = mip.BINARY) for ix in range(GamesInPlan)]
    slackVars = [tixModel.add_var(name = f"slack_game_{ix}+", var_type = mip.CONTINUOUS, ub = 0.0) for ix in range(GamesInPlan)]
    slackVars += [tixModel.add_var(name = f"slack_game_{ix}-", var_type = mip.CONTINUOUS, ub = 0.0) for ix in range(GamesInPlan)]

    # All tickets must be allocated

    for ix in range(GamesInPlan):
        tixModel.add_constr(mip.xsum(tixVars[ix + 2 * iy * GamesInPlan] + 2.0 * tixVars[ix + GamesInPlan + 2 * iy * GamesInPlan] for iy in range(len(fans))) + slackVars[2 * ix] - slackVars[2 * ix + 1] == schedule[ix].pairs, name = f"Allocate_All_Tickets_game_{ix}")

    # Each fan must attend the correct number of games + satisfy all personal constraints

    for iy, fan in enumerate(fans):
        for ix, constraint in enumerate(fan.constraints):
            for coefDict in constraint.gameDictionaries:
                slackVars += [tixModel.add_var(name = f"slack_{fan.name}_constraint{ix}+", var_type = mip.CONTINUOUS, ub = 0.0)]
                slackVars += [tixModel.add_var(name = f"slack_{fan.name}_constraint{ix}-", var_type = mip.CONTINUOUS, ub = 0.0)]
                linFunc = mip.xsum(count * tixVars[iz + iy * 2 * GamesInPlan] for iz, count in coefDict.items()) + slackVars[-2] - slackVars[-1]
                if constraint.comparator == '==':
                    tixModel.add_constr(linFunc == constraint.value, name = f"{fan.name}_constraint{ix}")
                if constraint.comparator == '<=':
                    tixModel.add_constr(linFunc <= constraint.value, name = f"{fan.name}_constraint{ix}")
                if constraint.comparator == '>=':
                    tixModel.add_constr(linFunc >= constraint.value, name = f"{fan.name}_constraint{ix}")

    # Establish the objective function

    costs = []
    for fan in fans:
        costs += fan.useRanking
    tixModel.objective = mip.xsum(costs[ix] * tixVars[ix] for ix in range(len(tixVars))) + mip.xsum(100000.0 * slackVars[ix] for ix in range(len(slackVars)))
except Exception as e:
    print(f"Mip setup exception: {e}")

In [ ]:
try:
    status = tixModel.optimize()
except Exception as e:
    print(f"Mip optimize exception: {e}")

if status != mip.OptimizationStatus.OPTIMAL:
    print(f"No solution found: {status}")
    if status == mip.OptimizationStatus.INFEASIBLE:
        print("Infeasible solution found.  Relaxing constraints to find a solution with minimum slack.")
        tixRelax = tixModel.copy()
        for var in tixRelax.vars:
            if var.name.startswith("slack"):
                var.ub = 1.0
        try:
            status = tixRelax.optimize(max_seconds = 60)
        except Exception as e:
            print(f"Mip relax optimize exception: {e}")
        print(status)
        violatedConstraints = []
        for var in tixRelax.vars:
            if var.name.startswith("slack") and (var.x is None or var.x > 0.0):
                violatedConstraints.append(var.name.removeprefix("slack_").removesuffix("+").removesuffix("-"))
        print("Violated constraints:  ", violatedConstraints)
        for constraint in tixRelax.constrs:
            if constraint.name in violatedConstraints:
                print(constraint)

In [ ]:
picks = [[] for fan in fans]
for mix, mvar in enumerate(tixModel.vars):
    if mvar.x is not None and mvar.x > 0.5:
        fix = mix // (2 * GamesInPlan)
        if len(fans[fix].ranking) > GamesInPlan:
            gix = mix % (2 * GamesInPlan)
        else:
            gix = mix % GamesInPlan
        picks[fix].append(fans[fix].ranking[gix])
for fix, fan in enumerate(fans):
    picks[fix].sort()
    picks[fix] = [int(pick) for pick in picks[fix]]
    print(fans[fix].name, picks[fix])

In [ ]:
costs = {}
gameAllocationTable = "Day | Date | Time | Opponent | Type | Seats |"
for ix in range(MaxPairsPerGame):
    gameAllocationTable += f" Pair{ix+1} |"
gameAllocationTable += "\n| :-: | :-: | :-: | :-: | :-: | :-: |"  + " :-: |" * MaxPairsPerGame + "\n"
for gix, game in enumerate(schedule):
    gameAllocationTable += f"| {game.weekday} | {game.month}/{game.day} | {game.time} | {game.opponent} | {game.type} | {game.seats} |"
    for mix, mvar in enumerate(tixModel.vars):
        if mix % GamesInPlan == gix and mvar.x is not None and mvar.x > 0.5:
            fix = mix // (2 * GamesInPlan)
            if len(fans[fix].ranking) > GamesInPlan and mix % (2 * GamesInPlan) >= GamesInPlan:
                gix += GamesInPlan
            cost = costs.get(fans[fix].name, 0.0)
            costs[fans[fix].name] = cost + 2.0 * game.price
            gameAllocationTable += f" {fans[fix].name} ({fans[fix].ranking[gix]}) |"
            if mix % (2 * GamesInPlan) >= GamesInPlan:
                costs[fans[fix].name] += 2.0 * game.price
                gameAllocationTable += " |"
    gameAllocationTable += " ❌ |" * (MaxPairsPerGame - game.pairs) + "\n"
display(Markdown(gameAllocationTable))

In [ ]:
amountOwedTable = "| | Amount owed | Paid |\n"
amountOwedTable += "| :- | -: | -: |\n"
for name, cost in sorted(costs.items()):
    amountOwedTable += f"| {name} | {locale.currency(cost, grouping=True)} | |\n"
display(Markdown(amountOwedTable))